# Oxford-IIIT Pet Auto-Researcher

This notebook is the orchestration layer for the reusable `pet_researcher` package. Use it to run staged experiments, inspect outputs in `./runs`, and only evaluate finalists on the held-out test split.

In [ ]:
from pet_researcher import ExperimentConfig, run_experiment, run_hierarchical_experiment, final_test_report

In [ ]:
baseline = ExperimentConfig(
    name="stage1_resnet18_baseline",
    backbone="resnet18",
    augmentation="none",
    fine_tune_scope="layer3_4",
)

resnet18_recipe = ExperimentConfig(
    name="stage2_resnet18_mild_aug",
    backbone="resnet18",
    augmentation="mild",
    fine_tune_scope="layer3_4",
    scheduler="cosine",
    label_smoothing=0.05,
    finetune_lr=3e-4,
)

resnet50_recipe = ExperimentConfig(
    name="stage3_resnet50_layer4",
    backbone="resnet50",
    augmentation="mild",
    fine_tune_scope="layer4",
    scheduler="cosine",
    label_smoothing=0.05,
)

baseline_result = run_experiment(baseline)
resnet18_result = run_experiment(resnet18_recipe)
resnet50_result = run_experiment(resnet50_recipe)

baseline_result, resnet18_result, resnet50_result

In [ ]:
router_config = ExperimentConfig(
    name="stage4_router_species",
    task="species",
    backbone="resnet18",
    augmentation="mild",
    fine_tune_scope="layer4",
    label_smoothing=0.05,
)

cat_config = ExperimentConfig(
    name="stage4_cat_specialist",
    task="breed",
    backbone="resnet18",
    augmentation="mild",
    fine_tune_scope="layer3_4",
    label_smoothing=0.05,
    species_filter=0,
)

dog_config = ExperimentConfig(
    name="stage4_dog_specialist",
    task="breed",
    backbone="resnet18",
    augmentation="mild",
    fine_tune_scope="layer3_4",
    label_smoothing=0.05,
    species_filter=1,
)

hierarchy_result = run_hierarchical_experiment(router_config, cat_config, dog_config)
hierarchy_result.val_metrics

In [ ]:
finalists = []
for result in sorted([baseline_result, resnet18_result, resnet50_result], key=lambda item: item.val_metrics["acc"], reverse=True)[:2]:
    cfg = ExperimentConfig.from_dict(result.config.to_dict())
    cfg.name = f"{cfg.name}_tta"
    cfg.tta = True
    cfg.source_run_name = result.config.name
    cfg.resume_if_available = True
    finalists.append(cfg)

final_report = final_test_report(finalists)
final_report